# Phase Annotation

Complex agents often work in distinct stages — first researching, then planning, then executing. Phase Annotation uses `async with` syntax to draw clear boundaries around these stages, automatically managing goal switching, context scoping, and execution tracing.

In this tutorial, we'll build a **content creation agent** that works in three phases: gathering research material (`sequential`), writing articles in a loop (`loop`), and handling an urgent edit request via a temporary context override (`snapshot`).

## Initialize

First, let's set up the LLM and define the tools our content creation agent will use.

In [ ]:
import os
from openai import AsyncOpenAI

api_key = os.environ.get("OPENAI_API_KEY", "your-api-key")
base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

In [ ]:
from bridgic.amphibious import AmphibiousAutoma, CognitiveContext, CognitiveWorker, think_unit
from bridgic.model import OpenAILlm

llm = OpenAILlm(
    model="gpt-4o-mini",
    api_key=api_key,
    base_url=base_url,
)

In [ ]:
from bridgic.core import tool

@tool(description="Search for information about a topic")
async def research_topic(topic: str) -> str:
    return f"Research results for '{topic}': Found 5 relevant sources with key insights."

@tool(description="Create an outline for an article")
async def create_outline(title: str, research: str) -> str:
    return f"Outline for '{title}': Introduction, 3 main sections, conclusion"

@tool(description="Write an article section based on outline and research")
async def write_section(title: str, section: str, notes: str) -> str:
    return f"Written section '{section}' for '{title}' ({len(notes)} chars of notes used)"

@tool(description="Edit an existing article to fix issues")
async def edit_article(title: str, edit_instructions: str) -> str:
    return f"Edited '{title}': Applied changes — {edit_instructions}"

We have four tools:

- **`research_topic`** — searches for information about a given topic.
- **`create_outline`** — creates a structured outline for an article.
- **`write_section`** — writes a section of an article based on an outline and research notes.
- **`edit_article`** — applies edits to an existing article.

## sequential — Linear Execution Phases

`self.sequential(goal="...")` groups steps into a linear phase. The `goal` parameter automatically updates the context, so the LLM knows what stage it's in. When the `async with` block exits, the phase is complete and the execution trace records it as a single structured node.

Let's build a content creator that moves through three sequential phases: research, outline, and write.

In [ ]:
class ContentCreator(AmphibiousAutoma[CognitiveContext]):
    researcher = think_unit(
        CognitiveWorker.inline("Research the topic and gather key information."),
        max_attempts=5,
    )
    outliner = think_unit(
        CognitiveWorker.inline("Create a structured outline based on the research."),
        max_attempts=3,
    )
    writer = think_unit(
        CognitiveWorker.inline("Write one section of the article based on the outline."),
        max_attempts=5,
    )

    async def on_agent(self, ctx: CognitiveContext):
        # Phase 1: Research (sequential)
        async with self.sequential(goal="Gather research material on the topic"):
            await self.researcher

        # Phase 2: Outline (sequential)
        async with self.sequential(goal="Create article outline from research"):
            await self.outliner

        # Phase 3: Write (sequential)
        async with self.sequential(goal="Write the article based on the outline"):
            await self.writer

In [ ]:
agent = ContentCreator(llm=llm)
result = await agent.arun(
    goal="Write an article about the future of renewable energy",
    tools=[research_topic, create_outline, write_section],
)
print(result)

Each `sequential` block creates a distinct phase in the execution trace. Inside the block, the LLM's context is automatically updated with the phase goal — so during the research phase, the LLM sees "Gather research material on the topic" as its current objective, not the original high-level goal.

When the block exits, the goal is restored and the trace records the phase as a single node with all its internal steps nested inside.

## loop — Repeated Execution with Pattern Detection

`self.loop(goal="...")` groups steps into a repeating phase. The framework can detect repeating patterns, which is useful for iterative tasks like processing multiple items from a list.

Let's build a multi-article writer that researches all topics first, then writes an article for each one in a loop.

In [ ]:
class MultiArticleWriter(AmphibiousAutoma[CognitiveContext]):
    researcher = think_unit(
        CognitiveWorker.inline("Research the given topic."),
        max_attempts=3,
    )
    writer = think_unit(
        CognitiveWorker.inline(
            "Write a complete short article on the current topic. "
            "Each article should be 2-3 paragraphs."
        ),
        max_attempts=5,
    )

    async def on_agent(self, ctx: CognitiveContext):
        async with self.sequential(goal="Research all topics"):
            await self.researcher

        async with self.loop(goal="Write an article for each topic"):
            await self.writer

In [ ]:
agent = MultiArticleWriter(llm=llm)
result = await agent.arun(
    goal="Write short articles about: solar energy, wind power, and hydroelectric dams",
    tools=[research_topic, write_section],
)
print(result)

The `loop` phase tells the framework that the enclosed steps will repeat. The LLM processes each topic in turn — writing one article, then moving to the next. The framework's pattern detection can recognize when the loop is cycling through a set of items, which helps with execution tracing and debugging.

Note the difference from `sequential`: a sequential phase runs once and exits, while a loop phase repeats until the LLM determines all items are processed.

## snapshot — Temporary Context Override

`self.snapshot(goal="...", **fields)` temporarily overrides context fields. When the `async with` block exits, the original values are restored. This is perfect for handling interruptions — the agent can switch to an urgent task and then seamlessly return to what it was doing before.

In [ ]:
class InterruptibleWriter(AmphibiousAutoma[CognitiveContext]):
    writer = think_unit(
        CognitiveWorker.inline("Write article content based on the current goal."),
        max_attempts=5,
    )
    editor = think_unit(
        CognitiveWorker.inline("Apply the requested edit to the specified article."),
        max_attempts=3,
    )

    async def on_agent(self, ctx: CognitiveContext):
        # Main writing phase
        async with self.sequential(goal="Write the main article"):
            await self.writer

        # Urgent interruption: edit a previous article
        async with self.snapshot(goal="URGENT: Fix the title of the solar energy article"):
            await self.editor
            # After this block, context.goal is restored to the original

        # Continue with next task (original goal restored)
        async with self.sequential(goal="Write the conclusion"):
            await self.writer

In [ ]:
agent = InterruptibleWriter(llm=llm)
result = await agent.arun(
    goal="Write a series of articles about renewable energy",
    tools=[research_topic, write_section, edit_article],
)
print(result)

The `snapshot` phase is the key differentiator here. When the agent enters the snapshot block, its goal is temporarily replaced with the urgent edit task. The editor think unit sees only this new goal and acts accordingly. When the block exits, the original goal ("Write a series of articles about renewable energy") is automatically restored, and the agent continues writing the conclusion as if the interruption never happened.

This is especially useful in real-world scenarios where agents need to handle priority changes, user interruptions, or error recovery without losing their place in a larger workflow.

## Nesting Phases

Phases can be nested for complex multi-stage execution flows. For example, a sequential phase can contain a loop, or a loop can contain a snapshot for handling edge cases on specific iterations.

In [ ]:
class NestedPhaseWriter(AmphibiousAutoma[CognitiveContext]):
    researcher = think_unit(
        CognitiveWorker.inline("Research the current topic."),
        max_attempts=3,
    )
    writer = think_unit(
        CognitiveWorker.inline("Write one section of the article."),
        max_attempts=5,
    )

    async def on_agent(self, ctx: CognitiveContext):
        async with self.sequential(goal="Research phase"):
            await self.researcher

        async with self.sequential(goal="Writing phase"):
            async with self.loop(goal="Write each section"):
                await self.writer

In this example, the outer `sequential` phase ("Writing phase") contains an inner `loop` phase ("Write each section"). The execution trace reflects this nesting — you'll see the loop node nested inside the sequential node, making it easy to understand the agent's behavior at any level of detail.

You can nest phases as deeply as needed. Each level adds structure to the execution trace and scopes the goal for the LLM.

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we explored the three types of Phase Annotation:

- **`self.sequential(goal="...")`** groups steps into a linear phase. The goal is automatically updated in the context, guiding the LLM's focus.
- **`self.loop(goal="...")`** groups steps into a repeating phase with built-in pattern detection. Ideal for iterative tasks like processing multiple items.
- **`self.snapshot(goal="...", **fields)`** temporarily overrides context fields. When the block exits, original values are restored — perfect for handling interruptions.
- Phases can be **nested** for complex multi-stage execution flows.
- Each phase automatically generates structured nodes in the **execution trace** (AgentTrace), making it easy to analyze agent behavior.

Phase Annotation brings structure to agent execution, turning a flat stream of actions into a well-organized, traceable workflow.